In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
import datetime
import joblib
import os

In [9]:
# 1. LOAD DỮ LIỆU
path = '../../../data/insurance.csv'
if os.path.exists(path):
    print("✅ Đường dẫn ĐÚNG! File đã tìm thấy.")
    df = pd.read_csv(path)
else:
    print("Đường dẫn SAI! Python không thấy file.")

✅ Đường dẫn ĐÚNG! File đã tìm thấy.


In [10]:
# 2. FEATURE ENGINEERING (Tạo biến mới như bạn đã phân tích)
def feature_engineering(df):
    temp_df = df.copy()
    conditions = [
        (temp_df['bmi'] < 18.5),
        (temp_df['bmi'] < 25),
        (temp_df['bmi'] < 30),
        (temp_df['bmi'] >= 30)
    ]
    choices = ['Underweight', 'Normal', 'Overweight', 'Obese']
    temp_df['bmi_category'] = np.select(conditions, choices, default='Normal')
    
    temp_df['obese_smoker'] = ((temp_df['bmi'] >= 30) & (temp_df['smoker'] == 'yes')).astype(int)
    return temp_df

df_fe = feature_engineering(df)

In [11]:
# 3. CHIA TÁCH DỮ LIỆU
X = df_fe.drop('charges', axis=1)
y = df_fe['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:

# 4. ĐỊNH NGHĨA BỘ TIỀN XỬ LÝ (PREPROCESSOR)
def get_preprocess(X):
    categorical_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
    numerical_cols = X.select_dtypes(include=['int64', 'float64', 'number']).columns.tolist()
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_cols),
            ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
        ]
    )
    return preprocessor

preprocessor = get_preprocess(X_train)

In [13]:
# 5. TIỀN XỬ LÝ VÀ TRAIN MODEL
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42))
])

full_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'bmi', 'children',
                                                   'obese_smoker']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['sex', 'smoker', 'region',
                                                   'bmi_category'])])),
                ('model', GradientBoostingRegressor(random_state=42))])

In [14]:
# 6. ĐÁNH GIÁ MÔ HÌNH
y_pred = full_pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("="*50)
print(f"{'KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (GRADIENT BOOSTING)':^50}")
print("="*50)
print(f"R2 Score (Trên tập Test): {r2:>15.4f}")
print(f"MAE (Sai số tuyệt đối): {f'{mae:,.2f}':>22}")
print(f"RMSE (Sai số căn bậc hai): {f'{rmse:,.2f}':>20}")
print("="*50)


   KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (GRADIENT BOOSTING)   
R2 Score (Trên tập Test):          0.8765
MAE (Sai số tuyệt đối):               2,475.81
RMSE (Sai số căn bậc hai):             4,379.32


In [ ]:
# 7. TRỰC QUAN HÓA
metadata = {
    "model_name": "Gradient Boosting Regressor",
    "version": "1.0",
    "date_created": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "metrics": {
        "r2": r2,
        "mae": mae,
        "rmse": rmse
    },
    "features": list(X_train.columns)
}

model_package = {
    "pipeline": full_pipeline,
    "metadata": metadata
}

joblib_path = '../../../models/gradient_boosting_regressor_pipeline.joblib'
joblib.dump(model_package, joblib_path)

print(f"\n✅ Đã lưu thành công Full Pipeline và Metadata tại: {joblib_path}")

FileNotFoundError: [Errno 2] No such file or directory: '../../models/model_package.joblib'